# DS2002 · ACC Reality Check Starter

**2026-12-18 · Fall 2026**  

---

## ACC Reality Check: team notebook

Copy this notebook into your team repository and build the project in it. The sections follow the deliverables in the capstone PDF, so a grader can find each part where they expect it. Replace every TODO with your own work, and delete this paragraph when you are done.

Your published API lives in its own folder in the repository (for example `api/main.py`), because PythonAnywhere runs a Python file and not a notebook. Section 6 of this notebook only calls that API and draws a chart from what it returns.

### 0. Team, sources, and setup

**TODO:** list the team members, the two source URLs, and the date of each pull you used for the final results.

In [30]:
team = {
    'name': 'HEJ',
    'members': ['Honor Dold', 'Evie McMahan', 'Julia McPhillips'],
    'urls': {
        'repo_url': 'https://github.com/eviemcmahan/ds2002-capstone',
        'database_url': 'TODO',
        'raw_data_folder': 'TODO'
    },
    'roles': {
        'ratings CSV and crosswalk': 'Evie',
        'ESPN pulls and box scores': 'Honor',
        'SQLite schema and queries': 'Julia',
        'API and PythonAnywhere': 'Evie',
        'Chart 4': 'Julia',
        'Presentation slides': 'Honor'
    },
    'meeting_time': 'After class on Monday',
}

data = {
    'torvik_ratings': 'https://barttorvik.com/2027_team_results.csv',
    'espn': {
        'scoreboard': 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard',
        'game_summary': 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary'
    }
}

pull_dates = {
    'torvik_ratings': [11-23-26],
    'espn_scoreboard': 'TODO'
}

### 1. Extract

**TODO:** pull the Torvik CSV and the ESPN scoreboard and summary JSON. Save every raw response to disk, and read from the saved copy when it exists so that a rerun does not call the source again.

**Torvik CSV Pull**

In [ ]:
import requests, time, pandas as pd
from pathlib import Path
from datetime import date

RETRY_STATUSES = {429, 500, 502, 503, 504}
HEADERS = {'User-Agent': 'DS2002-UVA-student'}
RAW_TORVIK = Path('data/raw/torvik') # File path to save csv to
RAW_TORVIK.mkdir(parents=True, exist_ok=True) # Creates folder if not there already

def fetch_with_retries(url, tries=4, base_delay=0.5):
    last = None
    for attempt in range(tries): # Attempt up to 4 times
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code == 200: # Request succeeded
                return r.text                      
            if r.status_code not in RETRY_STATUSES: # Unknown error
                raise RuntimeError(f'not retryable: {r.status_code} {r.text[:200]}')
            last = f'status {r.status_code}'
        except requests.RequestException as e:
            last = repr(e)
        delay = base_delay * (2 ** attempt)
        print(f'attempt {attempt + 1} failed ({last}); retrying in {delay}s')
        time.sleep(delay) # Pause before attempting again
    raise RuntimeError(f'gave up after {tries} attempts: {last}') # Still failed after 4 tries

today = date.today().isoformat() # Save today's date
path = RAW_TORVIK / f'torvik_{today}.csv' # Save csv using today's date

if path.exists(): # Today's has already been pulled 
    print('using saved copy:', path)
else: # No pull yet
    path.write_text(fetch_with_retries(data['torvik_ratings'])) # Download and write file
    print('downloaded and saved:', path)

torvik = pd.read_csv(path) # Read the pulled file
torvik['snapshot_date'] = today
print('shape:', torvik.shape)
print(list(torvik.columns))
torvik.head()


downloaded and saved: data/raw/torvik/torvik_2026-11-23.csv
shape: (365, 46)
['rank', 'team', 'conf', 'record', 'adjoe', 'oe Rank', 'adjde', 'de Rank', 'barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'adjt', 'snapshot_date']


,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt,snapshot_date
0,1,Houston,B12,6-0,124.446566,3,93.775633,3,0.962823,1,...,0.0,0,0,0.831933,0,2.444763,1,26,67.546525,2026-11-23
1,2,Illinois,B10,5-1,126.253344,1,96.573708,16,0.956136,2,...,0.0,0,0,0.836544,0,0.985450,20,121,73.521881,2026-11-23
2,3,Kansas,B12,5-1,122.663254,6,94.266935,4,0.953827,3,...,0.0,0,0,0.829944,0,1.296250,9,160,72.710550,2026-11-23
3,4,Florida,SEC,6-1,124.734512,2,96.255920,13,0.951687,4,...,0.0,0,0,0.857038,0,2.286045,2,67,68.484550,2026-11-23
4,5,Duke,ACC,4-2,119.857865,20,93.211274,1,0.947426,5,...,0.0,0,0,0.801265,0,-0.309882,88,337,71.553221,2026-11-23


We downloaded Torvik's team ratings CSV and saved the raw text to `data/raw/torvik/` with the date of the pull in the file name. If that day's file already exists, the cell reads the saved copy instead of calling the site again. There are 365 rows in the file, with one row per Division I team. We also added a `snapshot_date` column to store the snapshot date when we merge this raw data. The raw file has 45 columns (46 including the snapshot date), including rank, team, conference, record, and many more ratings. Each pull is different based on the daily ranking. On our first pull on November 23rd, for example, the top 5 teams were Houston, Illinois, Kansas, Florida, and Duke, respectively.

In [3]:
# TODO: ESPN scoreboard, one request per date

In [4]:
# TODO: ESPN game summaries, one request per finished game

**Check Saved Raw Data**

In [29]:
RAW = Path('data/raw')

torvik_files = sorted((RAW / 'torvik').glob('torvik_*.csv')) # Find any torvik files, no matter the date
score_files = sorted((RAW / 'espn' / 'scoreboard').glob('*.json')) # Find any ESPN scoreboard files
summary_files = sorted((RAW / 'espn' / 'summary').glob('*.json')) # Find any ESPN summary files

print('Torvik snapshots:', len(torvik_files))
print('ESPN scoreboard dates:', len(score_files))
print('ESPN game summaries:', len(summary_files))

if score_files:
    print('Scoreboard range :', score_files[0].stem, 'to', score_files[-1].stem) # Print dates of scoreboard files

Torvik snapshots: 1
ESPN scoreboard dates: 0
ESPN game summaries: 0


TODO: At the end, short write up about our final raw saved data counts.

### 2. Transform

**TODO:** one function or clearly labeled cell per step, each printing a row count.

In [5]:
# TODO: the team name crosswalk, with the unmatched names printed before and after

In [6]:
# TODO: games, with the Eastern game date derived from the UTC start time

In [7]:
# TODO: team_games, one row per team per game, with margin and possessions

In [8]:
# TODO: player_games

### 3. Load into SQLite

**TODO:** create the tables with primary keys, load them so that a rerun cannot duplicate rows, and print the row count of every table.

In [9]:
# TODO: schema

In [10]:
# TODO: load, then prove that a second run leaves the counts unchanged

### 4. Questions

Each question gets a SQL query run against your database, a chart if the PDF asks for one, and an answer written in full sentences that includes the number it rests on.

#### Q1. Record versus rating

In [11]:
# TODO: SQL

In [12]:
# TODO: chart 1

_Q1 answer._

#### Q2. Strength of the opponents so far

In [13]:
# TODO: SQL

In [14]:
# TODO: chart 2

_Q2 answer._

#### Q3. Close games and recent form

In [15]:
# TODO: SQL, including a window function for the rolling margin

In [16]:
# TODO: chart 3

_Q3 answer._

#### Q4. Pace and points per possession

In [17]:
# TODO: SQL

_Q4 answer._

#### Q5. Who carries each team

In [18]:
# TODO: SQL

_Q5 answer._

#### Q6. Does the rating predict the winner?

In [19]:
# TODO: SQL that uses the latest snapshot taken before each game

_Q6 answer, with the number of games it covers._

### 5. Decision log

**TODO:** every decision that changed the data, with the number of rows it affected.

In [20]:
# TODO

### 6. The published API

**TODO:** the live address, the list of routes, and chart 4, drawn from JSON returned by your own API.

In [21]:
API_URL = 'TODO'

# TODO: call your routes with requests and draw chart 4

### 7. Reflection

**TODO:** answer the five reflection questions from the PDF, with every member contributing.

_reflection here_

### 8. Presentation video

**TODO:** the link to your team's recorded presentation, which runs no more than eight minutes and includes every member. Put the same link in the README.

_video link here_